# Column Transformer

In [1]:
import numpy as np 
import pandas as pd 

from sklearn.impute import SimpleImputer # for filling the missing values 
from sklearn.preprocessing import OneHotEncoder # Encoding 
from sklearn.preprocessing import OrdinalEncoder # Encoding  

In [2]:
df = pd.read_csv('covid_toy.csv')

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.20, random_state=42)

In [7]:
x_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


# Manually type output

In [9]:
# Adding simple imputer to fever column 
si = SimpleImputer(strategy='mean')
x_train_fever = si.fit_transform(x_train[['fever']])

# also the test data 
x_test_fever = si.fit_transform(x_test[['fever']])

x_train_fever.shape

(80, 1)

In [10]:
# ordinal encoding ---> cough 
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])

# also the test data 
x_test_cough = oe.fit_transform(x_test[['cough']])

x_train_cough.shape

(80, 1)

In [12]:
# OneHotEncoding ---> Gender, City 
ohe = OneHotEncoder(drop='first', sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender', 'city']])

# Also the test data 
x_test_gender_city = ohe.fit_transform(x_test[['gender', 'city']])

x_train_gender_city.shape

(80, 4)

In [14]:
# Extracting Age 
x_train_age = x_train.drop(columns=['gender', 'fever', 'cough', 'city']).values 

# also the test data 
x_test_age = x_test.drop(columns=['gender', 'fever', 'cough', 'city']).values 

x_train_age.shape

(80, 1)

In [15]:
x_train_transformed = np.concatenate((x_train_age, x_train_fever, 
                                     x_train_gender_city, x_train_cough), axis=1)

In [16]:
x_train_transformed.shape

(80, 7)

# By the help of Column Transformer

In [19]:
from sklearn.compose import ColumnTransformer

In [22]:
transformer = ColumnTransformer(transformers = [
    # (constent, approch, column_name)
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']), 
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

# reminder -> passthrough =>  it means rest all the columns remain same

In [23]:
transformer.fit_transform(x_train).shape

(80, 7)

array([[101.,   0.,   0.,   0.,   0.,   1.,  81.],
       [100.,   0.,   0.,   0.,   1.,   0.,   5.],
       [100.,   0.,   0.,   0.,   1.,   0.,  19.],
       [100.,   0.,   1.,   1.,   0.,   0.,  27.],
       [103.,   0.,   0.,   1.,   0.,   0.,  73.],
       [103.,   1.,   1.,   0.,   1.,   0.,  70.],
       [102.,   0.,   0.,   1.,   0.,   0.,  49.],
       [101.,   1.,   0.,   0.,   1.,   0.,  51.],
       [101.,   0.,   0.,   1.,   0.,   0.,  64.],
       [101.,   0.,   0.,   0.,   1.,   0.,  83.],
       [ 98.,   0.,   0.,   0.,   0.,   1.,  65.],
       [104.,   0.,   0.,   0.,   0.,   0.,  18.],
       [103.,   0.,   0.,   0.,   0.,   0.,  16.],
       [104.,   0.,   1.,   0.,   1.,   0.,  16.],
       [100.,   0.,   1.,   0.,   1.,   0.,  27.],
       [101.,   0.,   0.,   0.,   0.,   0.,  84.],
       [104.,   0.,   1.,   0.,   1.,   0.,  51.],
       [102.,   0.,   0.,   0.,   0.,   0.,  69.],
       [102.,   1.,   0.,   0.,   0.,   0.,  82.],
       [103.,   0.,   0.,   0.,